# 《金玉良炎》講師出版工作台

**固定資料版｜計算與畫圖分離｜SVG／PNG／Word／PDF／互動 HTML**

> 完全虛構聲明：本 Notebook 與資料中的疾病、病原、檢驗、人物、場所、事件及數值均為合成教學內容，不對應任何真實歷史人物、作品情節、疾病或研究資料。

這份 Notebook 保留 canonical 標準資料，不包含資料生成或 seed 調整。講師可以在 `0.2` 的單一調整區改變分析與輸出選項；各幕先計算可核對的表格，再由同一份表格產生出版圖。最後會建立：

1. 八張直接標示數值的 `publication` 圖，每張同時輸出 SVG 與 PNG。
2. `chart_index.csv` 與 `chart_index.md` 圖表索引。
3. 可編輯的 Word 調查報告。
4. 適合列印的 PDF 調查報告。
5. 保留 hover、縮放與 SVG 下載功能的離線互動 HTML 延伸版。

## 在 Colab 的執行順序

- 執行 `0.1` 安裝套件與中文字型。
- 在 `0.2` 只修改標示為「講師可調整」的值。
- 執行 `1.1` 時上傳 `jinyuliang_student_release_1.zip`。
- 執行到 `8.1` 時依提示上傳 Release 2 與 Release 3。
- 依序執行到 `12.2`，最後下載報告與圖表資料夾。

`publication` 是本 Notebook 的主要模式；互動 Plotly 報告放在最後，作為延伸視野。


## 執行摘要

千秋宴後，御醫署先收到集中出現的嘔吐與腹瀉急報，再從固定活動名冊、食品暴露、菜單、檢驗與環境紀錄逐步還原事件。這份工作台採用「先固定分母、再建立病例、最後整合證據」的調查節奏：先確認誰能被判定，再看病例落在哪些人、哪些宮苑與哪些時間，最後才追問哪一項食品最可能是共同來源。

在預設 primary 病例定義下，312 名參與者中有 278 人可分類，辨認出 77 名主要病例。描述性結果顯示三處宮苑均有病例，流行曲線在宴後形成集中波峰；食品粗分析首先把翡翠凝露羹與合歡桂露飲帶入候選範圍。進一步檢查共食結構、場地分層與多變項模型後，翡翠凝露羹的訊號仍然保留，而合歡桂露飲的調整信賴區間包含無關聯值。缺失暴露的極端情境沒有改變主要方向，後續人體、食品與環境資料則用來檢查這條統計線索是否能與直接證據和候膳條件相互接合。

這是一個教學用的合成案件；報告中的 OR、陽性檢驗、食品分型與環境梯度都必須被理解為證據鏈中的不同節點，而不是任何單一數值就能獨立完成的結論。

## 分析 pipeline

急報與固定資料 → 病例定義與分母 → Person／Place／Time → 食品粗分析 → 共食結構 → 場地分層與多變項調整 → 缺失值敏感度 → 人體／食品／環境三角驗證 → 圖表、Word、PDF 與互動 HTML 交付。

每一幕都遵循同一個工作節奏：先建立可核對的結果表，再把結果畫成出版圖，最後用「本幕發現 → 下一步」把調查故事往前推進。


## 報告寫法參考

本工作台的敘事結構參考公共衛生疫情調查報告的常見順序：事件背景與調查啟動、調查方法、描述性流行病學、分析性流行病學、實驗室與環境調查、討論與限制，以及公共衛生行動。文字為本合成案件重新撰寫，不複製任何真實報告內容；方法上的核心原則是讓流行病學、實驗室、環境與食品安全證據在同一條調查線上彼此對照。

參考：

- [WHO｜Investigating foodborne disease outbreaks](https://www.who.int/publications/i/item/9789240118607)
- [CDC｜Multistate Foodborne Outbreaks: Investigation Steps](https://www.cdc.gov/foodborne-outbreaks/outbreak-basics/investigation-steps.html)


In [ ]:
# ─────────────────────────────────────────────────────────────
# 0.1｜安裝套件、中文字型與匯出工具
# 這格只需執行一次。fonts-noto-cjk 可避免 Matplotlib 中文顯示為方框。
# ─────────────────────────────────────────────────────────────
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run(
        ["apt-get", "update", "-qq"],
        check=True,
        stdout=subprocess.DEVNULL,
    )
    subprocess.run(
        ["apt-get", "install", "-y", "-qq", "fonts-noto-cjk"],
        check=True,
        stdout=subprocess.DEVNULL,
    )

# Plotly：互動 HTML；python-docx：Word；WeasyPrint：列印版 PDF。
%pip -q install "plotly>=6.1,<7" "python-docx>=1.1,<2" "weasyprint>=62,<70"

from pathlib import Path
from datetime import datetime
import base64
import io
import math
import zipfile

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import fisher_exact, norm

import matplotlib as mpl
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from matplotlib import font_manager

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", 80)
print("分析環境已就緒。")


## 0｜講師調整區

這一格是全份 Notebook 的控制面板。資料內容保持固定；可調整的是「怎麼定義、怎麼彙整、怎麼呈現與輸出」。初次試跑建議完全保留預設值。


### 講師敘事主線

請把每次參數實驗當成對同一案件的另一種詢問，而不是重新發明一個案件：病例定義改變的是「誰算進來」，流行曲線分箱改變的是「時間聚集如何被看見」，候選食品與調整因子改變的是「哪條來源解釋被比較」。每次修改後，都要重新讀取本幕的分母、圖說、本幕發現與下一步，確保故事仍然由資料推動。


In [ ]:
# ─────────────────────────────────────────────────────────────
# 0.2｜講師可調整區
# 修改等號右側即可；下方分析函式與繪圖函式通常不需要改。
# ─────────────────────────────────────────────────────────────

# 【病例定義】選 primary、sensitive 或 strict。
CASE_DEFINITION = "primary"

# 【病例時間窗】單位為餐後小時。可用來做病例定義敏感度實驗。
CASE_WINDOWS = {
    "primary": (4, 36),
    "sensitive": (3, 48),
    "strict": (5, 30),
}

# 【流行曲線】出版報告只放一個分箱；互動版保留多個分箱切換。
PUBLICATION_EPI_BIN_HOURS = 4
INTERACTIVE_EPI_BINS = [1, 2, 4, 6]

# 【候選食品】第一項是主要候選，第二項是粗分析中的共食候選。
CANDIDATE_FOODS = ["jade_dew_custard", "harmony_osmanthus_drink"]

# 【調整因子】本案例預設控制場地。若增刪欄位，必須先確認資料中存在。
ADJUSTMENT_FACTORS = ["site_id"]

# 【統計與缺失值】信賴水準與暴露缺失的三種情境。
CONFIDENCE_LEVEL = 0.95
MISSING_EXPOSURE_SCENARIOS = [
    "complete_case",
    "missing_as_unexposed",
    "missing_as_exposed",
]

# 【出版輸出】Word／PDF 使用 PNG；SVG 另存供排版與後製。
PUBLICATION_DPI = 220
EXPORT_SVG = True
EXPORT_PNG = True
BUILD_WORD = True
BUILD_PDF = True
BUILD_INTERACTIVE_HTML = True

# 【輸出位置】所有檔案集中在同一資料夾，方便最後整批下載。
OUTPUT_ROOT = Path("jinyuliang_instructor_outputs")
PUBLICATION_DIR = OUTPUT_ROOT / "publication_figures"
PUBLICATION_DIR.mkdir(parents=True, exist_ok=True)

print("目前病例定義：", CASE_DEFINITION)
print("出版流行曲線分箱：", PUBLICATION_EPI_BIN_HOURS, "小時")
print("候選食品：", CANDIDATE_FOODS)
print("輸出資料夾：", OUTPUT_ROOT.resolve())


In [ ]:
# ─────────────────────────────────────────────────────────────
# 0.3｜出版與互動圖的共用視覺設定
# ─────────────────────────────────────────────────────────────
COLORS = {
    "cinnabar": "#934631",
    "gold": "#B38A3E",
    "jade": "#2E6255",
    "ink": "#25352F",
    "paper": "#FBF7EE",
    "muted": "#746D63",
    "line": "#D9CDBA",
    "blue": "#58708C",
}

SITE_LABELS = {
    "zhaohua_hall": "昭華殿",
    "hengfang_court": "蘅芳苑",
    "chengyue_terrace": "承月臺",
}
ROLE_LABELS = {
    "court_attendant": "宮廷侍從",
    "inner_court_guest": "內廷賓客",
    "kitchen_service_staff": "膳房與供膳人員",
    "music_dance_staff": "樂舞人員",
    "palace_guard": "宮廷侍衛",
}

def configure_cjk_font():
    '''尋找 Noto CJK 字型並套用；若找不到就明確報錯，避免產生方框圖。'''
    font_candidates = [
        Path.cwd() / "assets" / "fonts" / "NotoSansCJK-Regular.ttc",
        Path("/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"),
        Path("/usr/share/fonts/opentype/noto/NotoSansCJKtc-Regular.otf"),
    ]
    font_path = next((path for path in font_candidates if path.exists()), None)
    if font_path is None:
        matches = list(Path("/usr/share/fonts").rglob("*NotoSansCJK*"))
        font_path = matches[0] if matches else None
    if font_path is None:
        raise RuntimeError(
            "找不到 Noto CJK 字型。請重新執行 0.1，確認 fonts-noto-cjk 安裝成功。"
        )
    font_manager.fontManager.addfont(str(font_path))
    family = font_manager.FontProperties(fname=str(font_path)).get_name()
    mpl.rcParams.update({
        "font.family": family,
        "axes.unicode_minus": False,
        "figure.facecolor": COLORS["paper"],
        "axes.facecolor": "white",
        "axes.edgecolor": COLORS["line"],
        "axes.labelcolor": COLORS["ink"],
        "text.color": COLORS["ink"],
        "xtick.color": COLORS["ink"],
        "ytick.color": COLORS["ink"],
        "axes.titleweight": "bold",
    })
    return font_path, family

CJK_FONT_PATH, CJK_FONT_FAMILY = configure_cjk_font()
PLOTLY_CJK_FONT = f"{CJK_FONT_FAMILY}, Noto Sans TC, Microsoft JhengHei, sans-serif"

def svg_config(filename):
    '''互動圖右上角的相機按鈕固定下載 SVG。'''
    return {
        "displaylogo": False,
        "responsive": True,
        "displayModeBar": "hover",
        "toImageButtonOptions": {
            "format": "svg",
            "filename": filename,
            "height": None,
            "width": None,
            "scale": 1,
        },
    }

def apply_plotly_theme(fig, title, height=520, x_title=None, y_title=None):
    '''互動 HTML 專用主題；與出版圖共用宮廷色票。'''
    fig.update_layout(
        template="plotly_white",
        title={"text": title, "x": 0.02, "xanchor": "left"},
        height=height,
        paper_bgcolor=COLORS["paper"],
        plot_bgcolor="#FFFFFF",
        font={"family": PLOTLY_CJK_FONT, "size": 14, "color": COLORS["ink"]},
        title_font={"family": PLOTLY_CJK_FONT, "size": 23, "color": COLORS["ink"]},
        margin={"l": 92, "r": 54, "t": 86, "b": 76},
        legend={"orientation": "h", "y": 1.04, "x": 1, "xanchor": "right"},
    )
    fig.update_xaxes(title=x_title, gridcolor="#E8E0D3", linecolor=COLORS["line"])
    fig.update_yaxes(title=y_title, gridcolor="#E8E0D3", linecolor=COLORS["line"])
    return fig

PUBLICATION_FIGURES = {}
INTERACTIVE_FIGURES = {}
CHART_INDEX = []

def save_publication_figure(fig, chart_id, stage, title, denominator, caption):
    '''同時輸出 SVG／PNG，並把用途與分母寫入圖表索引。'''
    # 預留上方標題／分母與下方圖說空間，避免座標標題和圖說擠在一起。
    fig.subplots_adjust(top=0.86, bottom=0.16)
    fig.suptitle(title, x=0.08, ha="left", fontsize=17, color=COLORS["ink"], fontweight="bold")
    fig.text(0.08, 0.935, denominator, ha="left", va="top", fontsize=9.5, color=COLORS["muted"])
    fig.text(0.08, 0.015, caption, ha="left", va="bottom", fontsize=8.8, color=COLORS["muted"])

    png_path = PUBLICATION_DIR / f"{chart_id}.png"
    svg_path = PUBLICATION_DIR / f"{chart_id}.svg"
    if EXPORT_PNG:
        # 先完整寫入記憶體再落盤，避免雲端磁碟同步時留下不完整 PNG。
        png_buffer = io.BytesIO()
        fig.savefig(png_buffer, format="png", dpi=PUBLICATION_DPI,
                    bbox_inches="tight", facecolor=COLORS["paper"])
        png_path.write_bytes(png_buffer.getvalue())
    if EXPORT_SVG:
        fig.savefig(svg_path, format="svg", bbox_inches="tight", facecolor=COLORS["paper"])
    PUBLICATION_FIGURES[chart_id] = fig
    CHART_INDEX.append({
        "chart_id": chart_id,
        "stage": stage,
        "title_zh": title,
        "denominator": denominator,
        "caption": caption,
        "png_file": str(png_path) if EXPORT_PNG else "",
        "svg_file": str(svg_path) if EXPORT_SVG else "",
        "publication_labels": "direct",
        "interactive_extension": "yes",
    })
    plt.show()
    return png_path, svg_path

print("出版中文字型：", CJK_FONT_FAMILY)


## 1｜固定資料：上傳、讀取與分母檢查

這一幕只處理資料來源與品質檢查。它建立活動名冊、食品暴露、菜單、檢驗及環境資料的讀取方式，不建立任何新的合成人物或暴露。

### 報告閱讀提示

先確認資料表的列數、欄數與外鍵關係，再進入病例分類。這裡的 `people` 是事件分母，`exposures` 是食品暴露資料；後續每一張圖都應能回溯到這兩張表。若資料筆數或 `person_id` 唯一性不符合預期，應先停止解讀圖表，而不是直接進入統計分析。


In [ ]:
# ─────────────────────────────────────────────────────────────
# 1.1｜尋找或上傳分階段資料
# ─────────────────────────────────────────────────────────────
def find_release_dir(release_name):
    '''優先讀取本機固定資料；Colab 找不到時才要求上傳對應 ZIP。'''
    candidates = [
        Path.cwd() / "student" / release_name,
        Path.cwd().parent / "student" / release_name,
        Path("/content/jinyuliang") / "student" / release_name,
    ]
    marker = "person_line_list.csv" if release_name == "release_1" else "README.md"
    for candidate in candidates:
        if (candidate / marker).exists():
            return candidate

    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(f"找不到 {release_name}；請提供對應 Release ZIP。") from exc

    print(f"請上傳 jinyuliang_student_{release_name}.zip")
    uploaded = files.upload()
    zip_name, zip_bytes = next(iter(uploaded.items()))
    destination = Path("/content/jinyuliang")
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as archive:
        archive.extractall(destination)
    matches = list(destination.rglob(marker))
    if not matches:
        raise FileNotFoundError(f"{zip_name} 中找不到 {marker}")
    return matches[0].parent

RELEASE_1 = find_release_dir("release_1")
people = pd.read_csv(RELEASE_1 / "person_line_list.csv", dtype={"person_id": "string"})
exposures = pd.read_csv(RELEASE_1 / "food_exposure.csv", dtype={"person_id": "string"})
menu = pd.read_csv(RELEASE_1 / "menu_by_site.csv")
dictionary = pd.read_csv(RELEASE_1 / "data_dictionary.csv")

for column in [
    "meal_start_datetime", "meal_end_datetime", "survey_datetime",
    "symptom_onset_datetime", "symptom_end_datetime",
]:
    people[column] = pd.to_datetime(people[column], errors="coerce")

dataset_summary = pd.DataFrame({
    "table": ["people", "exposures", "menu", "dictionary"],
    "rows": [len(people), len(exposures), len(menu), len(dictionary)],
    "columns": [people.shape[1], exposures.shape[1], menu.shape[1], dictionary.shape[1]],
    "source": [
        "Release 1 活動名冊與症狀問卷",
        "Release 1 食品暴露問卷",
        "Release 1 膳房供餐紀錄",
        "Release 1 欄位字典",
    ],
})
display(dataset_summary)

assert len(people) == 312, "固定標準資料應有 312 人。"
assert len(exposures) == 3120, "應為 312 人 × 10 項食品。"
assert people["person_id"].is_unique, "person_id 必須一人一號。"
assert set(exposures["person_id"]).issubset(set(people["person_id"])), "食品問卷外鍵錯誤。"


## 2｜病例定義：先計算，再選擇後續分析分母

`classify_cases()` 把《疾病工作手冊》的文字規則轉成三個可重跑欄位。問卷未回覆者保留為 `NA`，所以不會被默認為沒有生病。

### 報告閱讀提示

病例定義決定誰進入後續分母，也會影響每一張圖的病例數、侵襲率與效果估計。先同時列出 `primary`、`sensitive` 與 `strict` 的結果，再明確標示本次出版版採用的定義；這能把「病例規則造成的差異」與「食品或場地造成的差異」分開。


In [ ]:
# ─────────────────────────────────────────────────────────────
# 2.1｜計算病例定義（不畫圖）
# ─────────────────────────────────────────────────────────────
MAJOR_SYMPTOMS = ["nausea", "vomiting", "diarrhea", "abdominal_cramps"]

def classify_cases(data, windows):
    meal_end = pd.to_datetime(data["meal_end_datetime"], errors="coerce")
    onset = pd.to_datetime(data["symptom_onset_datetime"], errors="coerce")
    hours = (onset - meal_end).dt.total_seconds() / 3600
    symptoms = data[MAJOR_SYMPTOMS].apply(pd.to_numeric, errors="coerce")
    major_count = symptoms.sum(axis=1, min_count=1)
    vomiting_count = pd.to_numeric(data["vomiting_count_24h"], errors="coerce")
    stool_count = pd.to_numeric(data["loose_stool_count_24h"], errors="coerce")
    known = data["survey_response"].eq(1)

    primary = known & hours.between(*windows["primary"]) & (
        (major_count >= 2) | (vomiting_count >= 2) | (stool_count >= 3)
    )
    sensitive = known & hours.between(*windows["sensitive"]) & (major_count >= 1)
    strict = known & hours.between(*windows["strict"]) & (
        (symptoms["vomiting"].eq(1) | symptoms["diarrhea"].eq(1))
        & (symptoms["nausea"].eq(1) | symptoms["abdominal_cramps"].eq(1))
    )

    result = pd.DataFrame({"person_id": data["person_id"], "hours_after_meal": hours})
    for name, values in {"primary": primary, "sensitive": sensitive, "strict": strict}.items():
        status = pd.Series(pd.NA, index=data.index, dtype="Int64")
        status.loc[known] = values.loc[known].astype(int)
        result[name] = status
    return result

cases = classify_cases(people, CASE_WINDOWS)
analysis_people = people.merge(cases, on="person_id", validate="one_to_one")
case_summary = pd.DataFrame([
    {
        "case_definition": name,
        "classifiable_n": int(cases[name].notna().sum()),
        "cases": int(cases[name].sum(skipna=True)),
        "attack_rate": float(cases[name].mean()),
    }
    for name in ["primary", "sensitive", "strict"]
])
display(case_summary.style.format({"attack_rate": "{:.1%}"}))

assert CASE_DEFINITION in {"primary", "sensitive", "strict"}
assert int(cases["primary"].sum()) == 77
assert int(cases["primary"].notna().sum()) == 278


## 3｜Person／Place

病例名單建立後，先比較三處宮苑與五類角色的侵襲率。每個群組都以該群組中「可依所選病例定義判定的人」作分母。

### 圖說｜主要病例的 Person／Place 分布

左側 Place 面板比較三處宮苑的侵襲率，右側 Person 面板比較五類角色群組。每個標籤依序呈現「病例數／可分類分母」與侵襲率，因此應先確認分母，再比較柱子的高度。

**本圖想回答：** 病例是否集中在特定場地或特定角色群組，從而提供後續食品暴露與環境調查的方向。

**判讀提醒：** 群組差異是描述性訊號，不等於場地或角色本身造成疾病；群組大小、暴露機會、回答完整度與共食結構都可能影響觀察到的侵襲率。


In [ ]:
# ─────────────────────────────────────────────────────────────
# 3.1｜只計算 Person／Place 分析表
# ─────────────────────────────────────────────────────────────
def grouped_attack_rate(data, group_col, case_col):
    known = data.loc[data[case_col].notna()].copy()
    return (
        known.groupby(group_col, observed=True)
        .agg(classifiable_n=("person_id", "size"), cases=(case_col, "sum"))
        .assign(attack_rate=lambda x: x["cases"] / x["classifiable_n"])
        .reset_index()
    )

site_rates = grouped_attack_rate(analysis_people, "site_id", CASE_DEFINITION)
site_rates["label"] = site_rates["site_id"].map(SITE_LABELS)
role_rates = grouped_attack_rate(analysis_people, "role_group", CASE_DEFINITION)
role_rates["label"] = role_rates["role_group"].map(ROLE_LABELS)

display(site_rates.style.format({"attack_rate": "{:.1%}"}))
display(role_rates.style.format({"attack_rate": "{:.1%}"}))


In [ ]:
# ─────────────────────────────────────────────────────────────
# 3.2｜把既有表格畫成 publication 圖
# 數值直接放在圖上，適合 Word／PDF；不依賴 hover。
# ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.2), gridspec_kw={"wspace": 0.5})
for ax, frame, color, panel_title in [
    (axes[0], site_rates.sort_values("attack_rate"), COLORS["jade"], "Place｜三處宮苑"),
    (axes[1], role_rates.sort_values("attack_rate"), COLORS["cinnabar"], "Person｜角色群組"),
]:
    bars = ax.barh(frame["label"], frame["attack_rate"], color=color, alpha=0.94)
    ax.set_title(panel_title, loc="left", fontsize=12.5, pad=10)
    ax.set_xlabel("侵襲率")
    ax.xaxis.set_major_formatter(mpl.ticker.PercentFormatter(1))
    ax.grid(axis="x", color="#E8E0D3", linewidth=0.8)
    ax.set_axisbelow(True)
    ax.set_xlim(0, max(site_rates["attack_rate"].max(), role_rates["attack_rate"].max()) * 1.48)
    for bar, row in zip(bars, frame.itertuples()):
        ax.text(
            row.attack_rate + 0.008,
            bar.get_y() + bar.get_height() / 2,
            f"{int(row.cases)}/{int(row.classifiable_n)} · {row.attack_rate:.1%}",
            va="center", fontsize=9.5, color=COLORS["ink"],
        )
    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)

save_publication_figure(
    fig, "person_place_attack_rates", "A2",
    "主要病例的 Person／Place 分布",
    f"病例定義：{CASE_DEFINITION}；各群組分母為可分類者；全案可分類 n={int(cases[CASE_DEFINITION].notna().sum())}",
    "標籤依序顯示病例數／可分類分母與侵襲率；百分比不是全體病例構成比。",
)


## 4｜Time：流行曲線

三處宮苑都出現病例，下一步檢查病例是否集中在宴後相近時段。出版報告只放一個講師指定的分箱；互動延伸版再切換 1／2／4／6 小時。

### 圖說｜主要病例流行曲線

橫軸是症狀開始的日期與時間，縱軸是每個時間箱內的新發病例數；柱頂直接標示病例數。閱讀時先看病例是否形成集中波峰，再比較波峰與宴席、供餐或採檢時間的相對位置。

**本圖想回答：** 事件是否呈現共同來源型態，或是有較長時間的持續暴露與傳播可能。

**判讀提醒：** 分箱寬度會改變圖形的外觀，但不應改變病例總數；發病時間也可能受到回憶誤差、四捨五入與未回覆影響，因此流行曲線是線索，不是單獨的病因證據。


In [ ]:
# ─────────────────────────────────────────────────────────────
# 4.1｜只計算各分箱的病例數
# ─────────────────────────────────────────────────────────────
def epi_counts(data, bin_hours, case_col):
    selected = data.loc[
        data[case_col].eq(1) & data["symptom_onset_datetime"].notna(),
        "symptom_onset_datetime",
    ]
    start = selected.min().floor(f"{bin_hours}h")
    end = selected.max().ceil(f"{bin_hours}h")
    edges = pd.date_range(start, end + pd.Timedelta(hours=bin_hours), freq=f"{bin_hours}h")
    groups = pd.cut(selected, bins=edges, right=False)
    counts = groups.value_counts(sort=False)
    return pd.DataFrame({
        "bin_start": [interval.left for interval in counts.index],
        "bin_end": [interval.right for interval in counts.index],
        "cases": counts.to_numpy(),
    })

epi_tables = {
    hours: epi_counts(analysis_people, hours, CASE_DEFINITION)
    for hours in sorted(set(INTERACTIVE_EPI_BINS + [PUBLICATION_EPI_BIN_HOURS]))
}
publication_epi = epi_tables[PUBLICATION_EPI_BIN_HOURS]
expected_cases = int(cases[CASE_DEFINITION].sum(skipna=True))
assert all(int(table["cases"].sum()) == expected_cases for table in epi_tables.values())
display(publication_epi)


In [ ]:
# ─────────────────────────────────────────────────────────────
# 4.2｜publication 流行曲線
# 所有柱高直接標病例數；日期刻度由定位器控制，不把每個 bin 都塞進座標軸。
# ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12.5, 5.4))
width_days = PUBLICATION_EPI_BIN_HOURS / 24 * 0.92
bars = ax.bar(
    publication_epi["bin_start"], publication_epi["cases"],
    width=width_days, align="edge", color=COLORS["cinnabar"], edgecolor="white",
)
ax.bar_label(bars, labels=[str(int(value)) if value else "" for value in publication_epi["cases"]], padding=2, fontsize=8.5)
ax.set_xlabel("發病日期時間")
ax.set_ylabel("新發病例數")
ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=5, maxticks=9))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d\n%H:%M"))
ax.grid(axis="y", color="#E8E0D3", linewidth=0.8)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
ax.margins(x=0.01)

save_publication_figure(
    fig, "epidemic_curve", "A3",
    f"主要病例流行曲線｜{PUBLICATION_EPI_BIN_HOURS} 小時分箱",
    f"病例定義：{CASE_DEFINITION}；圖中病例總數 n={expected_cases}；每根柱為新發病例數",
    "分箱寬度會改變圖形外觀；所有版本使用同一批病例，柱高總和必須一致。",
)


## 5｜食品粗分析

時間分布與單次共同來源相容後，逐項比較食用與未食用者。每道食品可能有不同的未知回答，因此每項 OR 都要附上自己的完整案例分母。

### 圖說｜十項食品的粗 OR 與 95% CI

森林圖中的圓點是粗勝算比（OR），水平線是 95% 信賴區間，垂直虛線 OR=1 是無關聯參考線；右側表格同步列出 OR、CI 與該食品自己的完整案例分母。

**本圖想回答：** 哪些食品在未控制共食與場地前，呈現較大的病例差異，值得進入下一階段假說檢查。

**判讀提醒：** 粗 OR 只能描述食用者與未食用者的關聯，不能直接確認媒介；共食、場地混雜、暴露缺失與小分母都可能使區間變寬或估計偏移。


In [ ]:
# ─────────────────────────────────────────────────────────────
# 5.1｜只計算 2×2、OR、CI 與 Fisher exact test
# ─────────────────────────────────────────────────────────────
Z_CRITICAL = float(norm.ppf(1 - (1 - CONFIDENCE_LEVEL) / 2))

def odds_ratio_ci(a, b, c, d):
    corrected = any(value == 0 for value in [a, b, c, d])
    aa, bb, cc, dd = (a + 0.5, b + 0.5, c + 0.5, d + 0.5) if corrected else (a, b, c, d)
    estimate = aa * dd / (bb * cc)
    se = math.sqrt(1 / aa + 1 / bb + 1 / cc + 1 / dd)
    return (
        estimate,
        math.exp(math.log(estimate) - Z_CRITICAL * se),
        math.exp(math.log(estimate) + Z_CRITICAL * se),
        corrected,
    )

def two_by_two(data, exposure_col, outcome_col):
    x = pd.to_numeric(data[exposure_col], errors="coerce")
    y = pd.to_numeric(data[outcome_col], errors="coerce")
    valid = x.notna() & y.notna()
    x, y = x.loc[valid].astype(int), y.loc[valid].astype(int)
    a = int(((x == 1) & (y == 1)).sum())
    b = int(((x == 1) & (y == 0)).sum())
    c = int(((x == 0) & (y == 1)).sum())
    d = int(((x == 0) & (y == 0)).sum())
    estimate, low, high, corrected = odds_ratio_ci(a, b, c, d)
    _, p_value = fisher_exact([[a, b], [c, d]], alternative="two-sided")
    return {
        "n_complete": int(valid.sum()), "a": a, "b": b, "c": c, "d": d,
        "attack_rate_exposed": a / (a + b),
        "attack_rate_unexposed": c / (c + d),
        "odds_ratio": estimate, "ci_low": low, "ci_high": high,
        "fisher_p_value": p_value, "haldane_correction": corrected,
    }

exposure_case = exposures[["person_id", "food_id", "reported_consumed"]].merge(
    cases[["person_id", CASE_DEFINITION]], on="person_id", validate="many_to_one"
)
food_names = menu[["food_id", "food_name_zh"]].drop_duplicates("food_id")
food_results = pd.DataFrame([
    {"food_id": food_id, **two_by_two(group, "reported_consumed", CASE_DEFINITION)}
    for food_id, group in exposure_case.groupby("food_id", sort=False)
]).merge(food_names, on="food_id", validate="one_to_one")
food_results = food_results.sort_values("odds_ratio", ascending=False).reset_index(drop=True)

display(food_results.style.format({
    "attack_rate_exposed": "{:.1%}", "attack_rate_unexposed": "{:.1%}",
    "odds_ratio": "{:.2f}", "ci_low": "{:.2f}", "ci_high": "{:.2f}",
    "fisher_p_value": "{:.3g}",
}))


In [ ]:
# ─────────────────────────────────────────────────────────────
# 5.2｜publication 食品粗 OR 森林圖
# 左側是效果與 CI；右側把 OR、CI 與 n 直接印出，PDF 中不需 hover。
# ─────────────────────────────────────────────────────────────
plot_data = food_results.sort_values("odds_ratio").reset_index(drop=True)
y = np.arange(len(plot_data))
fig, (ax, table_ax) = plt.subplots(
    1, 2, figsize=(13.2, 7.2), gridspec_kw={"width_ratios": [3.2, 2.0], "wspace": 0.03}
)
ax.errorbar(
    plot_data["odds_ratio"], y,
    xerr=[plot_data["odds_ratio"] - plot_data["ci_low"], plot_data["ci_high"] - plot_data["odds_ratio"]],
    fmt="o", color=COLORS["cinnabar"], ecolor=COLORS["cinnabar"], capsize=4, markersize=7,
)
ax.axvline(1, color=COLORS["ink"], linestyle="--", linewidth=1)
ax.set_xscale("log")
ax.set_yticks(y, plot_data["food_name_zh"])
ax.set_xlabel("粗勝算比（log scale）")
ax.grid(axis="x", color="#E8E0D3", linewidth=0.8)
ax.spines[["top", "right", "left"]].set_visible(False)

table_ax.set_xlim(0, 1)
table_ax.set_ylim(-0.8, len(plot_data) - 0.2)
table_ax.axis("off")
table_ax.text(0.02, len(plot_data) - 0.05, "粗 OR（95% CI）", weight="bold", fontsize=10)
table_ax.text(0.82, len(plot_data) - 0.05, "n", weight="bold", fontsize=10, ha="right")
for i, row in plot_data.iterrows():
    table_ax.text(0.02, i, f"{row.odds_ratio:.2f}（{row.ci_low:.2f}–{row.ci_high:.2f}）", va="center", fontsize=9.2)
    table_ax.text(0.82, i, str(int(row.n_complete)), va="center", ha="right", fontsize=9.2)

save_publication_figure(
    fig, "food_crude_forest", "A4",
    f"十項食品的粗 OR 與 {CONFIDENCE_LEVEL:.0%} CI",
    f"病例定義：{CASE_DEFINITION}；每項食品使用自己的非缺失暴露分母 n",
    "OR=1 為無關聯參考線；粗 OR 用於篩選假說，後續仍需檢查共食、場地與直接證據。",
)


## 6｜共食結構

粗分析同時指出兩項食品時，先確認它們是否經常由同一批人一起食用。這張 heatmap 描述問卷暴露的成對相關，不代表任一食品是病因。

### 圖說｜十項食品的共食相關性

Heatmap 每一格是兩項食品暴露回答的成對 Pearson 相關係數，色彩與格內數值共同表示方向與強度；對角線固定為 1。重點不是找出「最紅」的一格，而是確認候選食品是否與其他食品形成共食群。

**本圖想回答：** 粗 OR 是否可能只是同一批人同時食用多項食品所造成的混雜訊號。

**判讀提醒：** 相關描述的是問卷中的共同暴露，不代表因果關係，也不表示相關較高的食品必然含有同一媒介；後續仍需用分層、模型與直接檢驗證據整合判讀。


In [ ]:
# ─────────────────────────────────────────────────────────────
# 6.1｜只計算食品共食相關矩陣
# ─────────────────────────────────────────────────────────────
exposure_wide = exposures.pivot(index="person_id", columns="food_id", values="reported_consumed")
corr = exposure_wide.apply(pd.to_numeric, errors="coerce").corr()
food_label_map = food_names.set_index("food_id")["food_name_zh"].to_dict()
ordered_ids = list(corr.columns)
ordered_labels = [food_label_map[food_id] for food_id in ordered_ids]
display(corr.round(2))
print("兩項候選食品相關係數：", round(float(corr.loc[CANDIDATE_FOODS[0], CANDIDATE_FOODS[1]]), 3))


In [ ]:
# ─────────────────────────────────────────────────────────────
# 6.2｜publication 共食 heatmap
# ─────────────────────────────────────────────────────────────
matrix = corr.loc[ordered_ids, ordered_ids].to_numpy()
fig, ax = plt.subplots(figsize=(10.6, 9.2))
image = ax.imshow(matrix, vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_xticks(np.arange(len(ordered_labels)), ordered_labels, rotation=38, ha="right")
ax.set_yticks(np.arange(len(ordered_labels)), ordered_labels)
for i in range(len(ordered_labels)):
    for j in range(len(ordered_labels)):
        value = matrix[i, j]
        ax.text(j, i, f"{value:.2f}", ha="center", va="center", fontsize=8.5,
                color="white" if abs(value) > 0.58 else COLORS["ink"])
colorbar = fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
colorbar.set_label("相關係數")
ax.spines[:].set_visible(False)

save_publication_figure(
    fig, "food_coexposure_heatmap", "A5",
    "十項食品的共食相關性",
    "分母：各食品配對中兩項暴露皆有回答者；格內為成對 Pearson 相關係數",
    "相關係數描述共食結構；它協助辨認混雜可能，不直接確認食品媒介。",
)


## 7｜分層與多變項調整

共食結構使粗 OR 無法拆開各食品的獨立訊號。這一幕以場地分層，並把兩項候選食品與指定調整因子放進同一個 logistic regression。

### 圖說｜候選食品的粗、分層與調整效果

森林圖並列同一候選食品在不同分析層次的 OR 與 95% CI：粗分析描述整體關聯，場地分層用來檢查不同宮苑中的一致性，調整模型則同時控制另一項候選食品與指定調整因子。

**本圖想回答：** 主要候選的訊號在控制共食與場地後是否仍保留，以及估計值改變的幅度。

**判讀提醒：** 調整後 OR 不是「真實效果」的自動保證；模型仍依賴病例定義、暴露測量、樣本數與調整變項的合理性。信賴區間若跨過 1，應以不確定性語言描述，而不是只看點估計。


In [ ]:
# ─────────────────────────────────────────────────────────────
# 7.1｜只計算 MH OR、logistic regression 與診斷
# ─────────────────────────────────────────────────────────────
def fit_logistic_mle(design, outcome):
    X = np.column_stack([np.ones(len(design)), design.to_numpy(dtype=float)])
    y = outcome.to_numpy(dtype=float)
    names = ["intercept"] + list(design.columns)

    def objective(beta):
        eta = X @ beta
        return float(np.logaddexp(0, eta).sum() - y @ eta)

    def gradient(beta):
        eta = np.clip(X @ beta, -35, 35)
        probability = 1 / (1 + np.exp(-eta))
        return X.T @ (probability - y)

    result = minimize(objective, np.zeros(X.shape[1]), jac=gradient, method="BFGS", options={"maxiter": 2000})
    beta = result.x
    probability = 1 / (1 + np.exp(-np.clip(X @ beta, -35, 35)))
    weights = probability * (1 - probability)
    information = X.T @ (X * weights[:, None])
    covariance = np.linalg.pinv(information)
    se = np.sqrt(np.diag(covariance))
    z = beta / se
    table = pd.DataFrame({
        "term": names, "coefficient": beta, "standard_error": se,
        "odds_ratio": np.exp(beta),
        "ci_low": np.exp(beta - Z_CRITICAL * se),
        "ci_high": np.exp(beta + Z_CRITICAL * se),
        "p_value": 2 * norm.sf(np.abs(z)),
    })
    diagnostics = {
        "converged": bool(result.success or np.linalg.norm(gradient(beta), ord=np.inf) < 1e-5),
        "n_observations": len(y), "n_cases": int(y.sum()),
        "max_abs_coefficient": float(np.abs(beta).max()),
        "information_condition_number": float(np.linalg.cond(information)),
    }
    return table, diagnostics

model_columns = ["person_id", CASE_DEFINITION] + ADJUSTMENT_FACTORS
model_data = analysis_people[model_columns].merge(
    exposure_wide[CANDIDATE_FOODS].reset_index(), on="person_id", validate="one_to_one"
).dropna(subset=[CASE_DEFINITION] + CANDIDATE_FOODS + ADJUSTMENT_FACTORS)

design_parts = [model_data[CANDIDATE_FOODS].astype(float)]
for factor in ADJUSTMENT_FACTORS:
    if not pd.api.types.is_numeric_dtype(model_data[factor]):
        design_parts.append(pd.get_dummies(model_data[factor], prefix=factor, drop_first=True, dtype=float))
    else:
        design_parts.append(model_data[[factor]].astype(float))
design = pd.concat(design_parts, axis=1)
adjusted, diagnostics = fit_logistic_mle(design, model_data[CASE_DEFINITION].astype(int))

# 第一項候選食品的場地分層 Mantel–Haenszel OR。
candidate_1, candidate_2 = CANDIDATE_FOODS[:2]
stratified_results = pd.DataFrame([
    {"site_id": site_id, **two_by_two(group, candidate_1, CASE_DEFINITION)}
    for site_id, group in model_data.groupby("site_id", sort=True)
])
n_i = stratified_results[["a", "b", "c", "d"]].sum(axis=1).astype(float)
a_i, b_i = stratified_results["a"].astype(float), stratified_results["b"].astype(float)
c_i, d_i = stratified_results["c"].astype(float), stratified_results["d"].astype(float)
r_i, s_i = a_i * d_i / n_i, b_i * c_i / n_i
R, S = r_i.sum(), s_i.sum()
mh_or = R / S
p_i, q_i = (a_i + d_i) / n_i, (b_i + c_i) / n_i
var_log_mh = (
    (p_i * r_i).sum() / (2 * R**2)
    + (p_i * s_i + q_i * r_i).sum() / (2 * R * S)
    + (q_i * s_i).sum() / (2 * S**2)
)
mh_se = math.sqrt(var_log_mh)
mh_low = math.exp(math.log(mh_or) - Z_CRITICAL * mh_se)
mh_high = math.exp(math.log(mh_or) + Z_CRITICAL * mh_se)

crude_lookup = food_results.set_index("food_id")
adjusted_lookup = adjusted.set_index("term")
effect_plot = pd.DataFrame([
    {"label": f"{food_label_map[candidate_1]}｜粗 OR", **crude_lookup.loc[candidate_1, ["odds_ratio", "ci_low", "ci_high"]].to_dict()},
    {"label": f"{food_label_map[candidate_1]}｜場地分層 MH OR", "odds_ratio": mh_or, "ci_low": mh_low, "ci_high": mh_high},
    {"label": f"{food_label_map[candidate_1]}｜調整 OR", **adjusted_lookup.loc[candidate_1, ["odds_ratio", "ci_low", "ci_high"]].to_dict()},
    {"label": f"{food_label_map[candidate_2]}｜粗 OR", **crude_lookup.loc[candidate_2, ["odds_ratio", "ci_low", "ci_high"]].to_dict()},
    {"label": f"{food_label_map[candidate_2]}｜調整 OR", **adjusted_lookup.loc[candidate_2, ["odds_ratio", "ci_low", "ci_high"]].to_dict()},
])
display(adjusted.style.format({"odds_ratio": "{:.2f}", "ci_low": "{:.2f}", "ci_high": "{:.2f}", "p_value": "{:.3g}"}))
display(pd.Series(diagnostics, name="model_diagnostics").to_frame())
assert diagnostics["converged"] and diagnostics["max_abs_coefficient"] < 8


In [ ]:
# ─────────────────────────────────────────────────────────────
# 7.2｜publication 調整前後效果比較
# ─────────────────────────────────────────────────────────────
plot_data = effect_plot.iloc[::-1].reset_index(drop=True)
y = np.arange(len(plot_data))
point_colors = [COLORS["blue"] if food_label_map[candidate_2] in label else COLORS["cinnabar"] for label in plot_data["label"]]
fig, (ax, table_ax) = plt.subplots(
    1, 2, figsize=(12.8, 5.5), gridspec_kw={"width_ratios": [3.2, 2.1], "wspace": 0.03}
)
for i, row in plot_data.iterrows():
    ax.errorbar(row.odds_ratio, i, xerr=[[row.odds_ratio-row.ci_low], [row.ci_high-row.odds_ratio]],
                fmt="o", color=point_colors[i], ecolor=point_colors[i], capsize=4, markersize=8)
ax.axvline(1, color=COLORS["ink"], linestyle="--", linewidth=1)
ax.set_xscale("log")
ax.set_yticks(y, plot_data["label"])
ax.set_xlabel("勝算比（log scale）")
ax.grid(axis="x", color="#E8E0D3", linewidth=0.8)
ax.spines[["top", "right", "left"]].set_visible(False)
table_ax.set_xlim(0, 1); table_ax.set_ylim(-0.8, len(plot_data)-0.2); table_ax.axis("off")
table_ax.text(0.02, len(plot_data)-0.05, "OR（95% CI）", weight="bold", fontsize=10)
for i, row in plot_data.iterrows():
    table_ax.text(0.02, i, f"{row.odds_ratio:.2f}（{row.ci_low:.2f}–{row.ci_high:.2f}）", va="center", fontsize=9.5)

save_publication_figure(
    fig, "adjusted_effect_comparison", "A6",
    "候選食品的粗、場地分層與調整效果",
    f"病例定義：{CASE_DEFINITION}；logistic regression 完整案例 n={diagnostics['n_observations']}，病例 n={diagnostics['n_cases']}",
    "粗、分層與調整效果回答不同問題；調整 OR 仍需與檢驗及環境證據整合。",
)


## 8｜暴露缺失敏感度

主要候選浮現後，固定病例分類，只改變第一項候選食品的未知暴露編碼。三個情境用來檢查效果方向是否依賴缺失處理。

### 圖說｜主要候選的暴露缺失敏感度

三列分別代表完整案例、未知全視為未食用、未知全視為食用；每列顯示一個 OR 與 95% CI。這不是在替未知回答猜測唯一真值，而是在兩個極端假設下觀察結論是否穩健。

**本圖想回答：** 主要候選的關聯方向與量級，是否只在某一種缺失值編碼下才成立。

**判讀提醒：** 極端情境的結果是敏感度範圍，不應被當成正式的替代分析；若不同情境方向一致，代表結論對此項缺失處理較穩健，但仍不能消除非隨機缺失的疑慮。


In [ ]:
# ─────────────────────────────────────────────────────────────
# 8.1｜只計算三種缺失情境
# ─────────────────────────────────────────────────────────────
base = exposures.loc[
    exposures["food_id"].eq(candidate_1), ["person_id", "reported_consumed"]
].merge(cases[["person_id", CASE_DEFINITION]], on="person_id", validate="one_to_one")
base = base.loc[base[CASE_DEFINITION].notna()].copy()
reported = pd.to_numeric(base["reported_consumed"], errors="coerce")
scenario_values = {
    "complete_case": reported,
    "missing_as_unexposed": reported.fillna(0),
    "missing_as_exposed": reported.fillna(1),
}
scenario_labels = {
    "complete_case": "完整案例",
    "missing_as_unexposed": "未知全視為未食用",
    "missing_as_exposed": "未知全視為食用",
}
missing_sensitivity = pd.DataFrame([
    {"assumption": name, "label": scenario_labels[name], **two_by_two(
        pd.DataFrame({"exposure": scenario_values[name], "outcome": base[CASE_DEFINITION]}),
        "exposure", "outcome",
    )}
    for name in MISSING_EXPOSURE_SCENARIOS
])
display(missing_sensitivity.style.format({"odds_ratio": "{:.2f}", "ci_low": "{:.2f}", "ci_high": "{:.2f}"}))


In [ ]:
# ─────────────────────────────────────────────────────────────
# 8.2｜publication 缺失值敏感度森林圖
# ─────────────────────────────────────────────────────────────
plot_data = missing_sensitivity.iloc[::-1].reset_index(drop=True)
y = np.arange(len(plot_data))
fig, (ax, table_ax) = plt.subplots(
    1, 2, figsize=(12.3, 4.7), gridspec_kw={"width_ratios": [3.1, 2.2], "wspace": 0.03}
)
ax.errorbar(plot_data["odds_ratio"], y,
            xerr=[plot_data["odds_ratio"]-plot_data["ci_low"], plot_data["ci_high"]-plot_data["odds_ratio"]],
            fmt="o", color=COLORS["jade"], ecolor=COLORS["jade"], capsize=4, markersize=8)
ax.axvline(1, color=COLORS["ink"], linestyle="--", linewidth=1)
ax.set_xscale("log"); ax.set_yticks(y, plot_data["label"]); ax.set_xlabel("勝算比（log scale）")
ax.grid(axis="x", color="#E8E0D3", linewidth=0.8); ax.spines[["top", "right", "left"]].set_visible(False)
table_ax.set_xlim(0, 1); table_ax.set_ylim(-0.8, len(plot_data)-0.2); table_ax.axis("off")
table_ax.text(0.02, len(plot_data)-0.05, "OR（95% CI）", weight="bold", fontsize=10)
table_ax.text(0.88, len(plot_data)-0.05, "n", weight="bold", ha="right", fontsize=10)
for i, row in plot_data.iterrows():
    table_ax.text(0.02, i, f"{row.odds_ratio:.2f}（{row.ci_low:.2f}–{row.ci_high:.2f}）", va="center", fontsize=9.3)
    table_ax.text(0.88, i, str(int(row.n_complete)), va="center", ha="right", fontsize=9.3)

save_publication_figure(
    fig, "missing_exposure_sensitivity", "A7",
    f"{food_label_map[candidate_1]}：暴露缺失敏感度分析",
    f"病例定義：{CASE_DEFINITION}；三個情境固定病例分類，只改未知暴露的編碼",
    "極端情境用於檢查結論穩健性，不代表未知值的真實狀態。",
)


## 9｜人體、食品與環境證據

觀察性分析提出主要食品候選後，Release 2 與 Release 3 分別加入人體檢驗、食品分型及候膳環境紀錄。陽性與陰性結果都要連同採檢時機、檢驗名稱與來源一起解讀。

### 圖說｜從關聯走向證據三角驗證

人體檢驗圖依採檢時間窗與檢驗方法呈現陽性數／檢驗數，環境圖則把各宮苑的暴露侵襲率、候膳溫度與保留時間放在同一個視覺框架中。兩張圖要一起讀：前者描述人體與食品檢驗的直接證據，後者提供場地與保存條件的脈絡。

**本圖想回答：** 人體、食品與環境資料是否指向同一個可解釋的來源鏈，而不是只依賴單一統計關聯。

**判讀提醒：** 陽性結果受採檢時機、檢驗方法與樣本來源影響；環境梯度與病例差異也可能由其他場地因素造成。三角驗證可以提高解釋的一致性，但仍須清楚區分「群聚成立」、「疾病身分」與「食品媒介確認」。


In [ ]:
# ─────────────────────────────────────────────────────────────
# 9.1｜只計算檢驗採檢時機與環境三角驗證表
# ─────────────────────────────────────────────────────────────
RELEASE_2 = find_release_dir("release_2")
RELEASE_3 = find_release_dir("release_3")
labs_r2 = pd.read_csv(RELEASE_2 / "lab_results.csv", dtype={"person_id": "string"})
labs_r3 = pd.read_csv(RELEASE_3 / "lab_results.csv", dtype={"person_id": "string"})
environment = pd.read_csv(RELEASE_3 / "environment_log.csv")
labs_r2["collection_datetime"] = pd.to_datetime(labs_r2["collection_datetime"], errors="coerce")
environment["record_datetime"] = pd.to_datetime(environment["record_datetime"], errors="coerce")

labs_person = labs_r2.merge(
    analysis_people[["person_id", "symptom_onset_datetime", CASE_DEFINITION]],
    on="person_id", how="left", validate="many_to_one",
)
labs_person["hours_after_onset"] = (
    labs_person["collection_datetime"] - labs_person["symptom_onset_datetime"]
).dt.total_seconds() / 3600
labs_person["sampling_window"] = pd.cut(
    labs_person["hours_after_onset"], [-np.inf, 24, 48, 72, np.inf],
    labels=["0–24 小時", "24–48 小時", "48–72 小時", "超過 72 小時"],
)
lab_summary = (
    labs_person.assign(positive=labs_person["result"].eq("positive"))
    .groupby(["test_name", "sampling_window"], observed=True)
    .agg(tests=("sample_id", "size"), persons=("person_id", "nunique"), positives=("positive", "sum"))
    .assign(positive_fraction=lambda x: x["positives"] / x["tests"])
    .reset_index()
)

positive_evidence = labs_r3.loc[labs_r3["result"].eq("positive"), [
    "source_type", "person_id", "food_id", "site_id", "test_name", "target_name", "subtype"
]].sort_values(["target_name", "source_type"])

candidate_exposure = exposures.loc[
    exposures["food_id"].eq(candidate_1), ["person_id", "reported_consumed"]
].copy()
candidate_exposure["reported_consumed"] = pd.to_numeric(candidate_exposure["reported_consumed"], errors="coerce")
site_candidate = analysis_people[["person_id", "site_id", CASE_DEFINITION]].merge(
    candidate_exposure, on="person_id", validate="one_to_one"
)
site_candidate = site_candidate.loc[
    site_candidate[CASE_DEFINITION].notna() & site_candidate["reported_consumed"].eq(1)
]
site_attack = (
    site_candidate.groupby("site_id")
    .agg(exposed_n=("person_id", "size"), cases=(CASE_DEFINITION, "sum"))
    .assign(attack_rate=lambda x: x["cases"] / x["exposed_n"])
    .reset_index()
)
environment_summary = (
    environment.loc[environment["food_id"].eq(candidate_1)]
    .groupby("site_id")
    .agg(median_temperature_c=("temperature_c", "median"),
         holding_minutes=("holding_duration_minutes", "median"), records=("log_id", "size"))
    .reset_index()
)
triangle = site_attack.merge(environment_summary, on="site_id", validate="one_to_one")
triangle["site_name_zh"] = triangle["site_id"].map(SITE_LABELS)

display(lab_summary.style.format({"positive_fraction": "{:.1%}"}))
display(positive_evidence)
display(triangle.style.format({"attack_rate": "{:.1%}", "median_temperature_c": "{:.1f}"}))


In [ ]:
# ─────────────────────────────────────────────────────────────
# 9.2｜publication 人體檢驗圖
# ─────────────────────────────────────────────────────────────
test_names = list(lab_summary["test_name"].drop_duplicates())
# 出版圖只保留實際有檢驗的時間窗，避免出現沒有分母的空白群組。
observed_windows = set(lab_summary["sampling_window"].astype(str))
windows = [
    str(value) for value in labs_person["sampling_window"].cat.categories
    if str(value) in observed_windows
]
x = np.arange(len(windows)); bar_width = 0.78 / max(1, len(test_names))
fig, ax = plt.subplots(figsize=(12.3, 5.6))
for index, test_name in enumerate(test_names):
    frame = lab_summary.loc[lab_summary["test_name"].eq(test_name)].set_index("sampling_window").reindex(windows)
    positions = x - 0.39 + bar_width / 2 + index * bar_width
    bars = ax.bar(positions, frame["positive_fraction"].fillna(0), width=bar_width,
                  label=test_name, color=[COLORS["cinnabar"], COLORS["jade"], COLORS["blue"]][index % 3])
    for bar, (_, row) in zip(bars, frame.iterrows()):
        if pd.notna(row.get("tests")):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.025,
                    f"{int(row.positives)}/{int(row.tests)}\n{int(row.persons)} 人",
                    ha="center", va="bottom", fontsize=8)
ax.set_xticks(x, windows); ax.set_xlabel("相對發病時間"); ax.set_ylabel("陽性比例")
ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1)); ax.set_ylim(0, 1.16)
ax.grid(axis="y", color="#E8E0D3", linewidth=0.8); ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False); ax.legend(frameon=False, ncol=3)

save_publication_figure(
    fig, "lab_sampling_summary", "A8",
    "人體檢驗陽性比例與採檢時機",
    "每根柱直接標示陽性檢驗數／檢驗數及受檢人數；同一人可能接受不同檢驗",
    "檢驗結果需與採檢時機、檢驗方法及病例狀態一起判讀；陰性不等同排除感染。",
)


In [ ]:
# ─────────────────────────────────────────────────────────────
# 9.3｜publication 場地環境三角驗證圖
# ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11.8, 5.9))
sizes = triangle["holding_minutes"] * 3.2
ax.scatter(triangle["median_temperature_c"], triangle["attack_rate"], s=sizes,
           c=[COLORS["cinnabar"], COLORS["gold"], COLORS["jade"]], alpha=0.88,
           edgecolor="white", linewidth=1.4)
for row in triangle.itertuples():
    ax.annotate(
        f"{row.site_name_zh}\n{row.cases:.0f}/{row.exposed_n:.0f} · {row.attack_rate:.1%}\n{row.median_temperature_c:.1f}°C · {row.holding_minutes:.0f} 分",
        (row.median_temperature_c, row.attack_rate), xytext=(0, 18), textcoords="offset points",
        ha="center", va="bottom", fontsize=9.2,
    )
ax.set_xlabel("候膳溫度中位數（°C）"); ax.set_ylabel("暴露者侵襲率")
ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1)); ax.set_ylim(0, max(0.63, triangle["attack_rate"].max()*1.35))
ax.grid(color="#E8E0D3", linewidth=0.8); ax.set_axisbelow(True); ax.spines[["top", "right"]].set_visible(False)

save_publication_figure(
    fig, "environment_triangulation", "A8",
    f"{food_label_map[candidate_1]}：場地環境三角驗證",
    "各點為一處宮苑；點大小代表候膳時間；標籤為病例／暴露分母、侵襲率、溫度與候膳時間",
    "環境梯度與食品暴露關聯相互支持；媒介確認仍依賴人體、食品與環境證據的一致性。",
)


## 10｜匯出 SVG、PNG 與圖表索引

前面每張 `publication` 圖已在繪製當下輸出。這一格把用途、分母、圖說與檔案位置整理成機器可讀 CSV 和講師可讀 Markdown。

### 報告閱讀提示

圖表索引是整份報告的目錄與品質檢查表：每一列對應一張出版圖，並保留圖名、分析階段、分母、圖說與 PNG／SVG 路徑。正式交付前，先確認圖表數量、檔案格式與分母文字一致，再組裝 Word、PDF 與互動版。


In [ ]:
# ─────────────────────────────────────────────────────────────
# 10.1｜建立圖表索引
# ─────────────────────────────────────────────────────────────
chart_index = pd.DataFrame(CHART_INDEX).drop_duplicates("chart_id", keep="last")
chart_index_csv = OUTPUT_ROOT / "chart_index.csv"
chart_index_md = OUTPUT_ROOT / "chart_index.md"
chart_index.to_csv(chart_index_csv, index=False, encoding="utf-8")

markdown_lines = [
    "# 金玉良炎圖表索引",
    "",
    "> 所有疾病、人物、場所、事件與數值均為完全虛構的合成教學內容。",
    "",
    "| 幕次 | chart_id | 圖表 | 分母／分析對象 | PNG | SVG |",
    "|---|---|---|---|---|---|",
]
for row in chart_index.itertuples():
    markdown_lines.append(
        f"| {row.stage} | `{row.chart_id}` | {row.title_zh} | {row.denominator} | `{row.png_file}` | `{row.svg_file}` |"
    )
chart_index_md.write_text("\n".join(markdown_lines) + "\n", encoding="utf-8")
display(chart_index)
print("圖表索引：", chart_index_csv.resolve())
print("出版圖數量：", len(chart_index))
assert len(chart_index) == 8


## 11｜組裝 Word 與列印版 PDF

Word 與 PDF 使用同一套「前因 → 此刻問題 → 圖表 → 本幕發現 → 下一步」內容。Word 方便修改；PDF 使用同一批 PNG 出版圖，數值不依賴 hover。

### 報告閱讀提示

這裡不是重新分析，而是把已核對的結果轉成可交付文件。閱讀報告時，先看每幕的問題，再看圖表中的直接標籤，最後讀「本幕發現」與「下一步」；這個順序能避免把單一顯著 OR 誤讀成完整的因果結論。


In [ ]:
# ─────────────────────────────────────────────────────────────
# 11.1｜建立連貫的報告內容
# 這裡只組裝敘事資料；下一格才寫入 Word／PDF。
# ─────────────────────────────────────────────────────────────
primary_row = case_summary.set_index("case_definition").loc[CASE_DEFINITION]
jade = food_results.set_index("food_id").loc[candidate_1]
osmanthus = food_results.set_index("food_id").loc[candidate_2]
adj = adjusted.set_index("term")

REPORT_SECTIONS = [
    {
        "stage": "A1", "title": "建立事件分母與病例名單", "chart_id": None,
        "pipeline": "事件急報 → 固定 Release 1 → 檢查 312 人名冊、食品暴露與資料完整性 → 產生三種病例狀態",
        "cause": "千秋宴結束後，宮中先收到多名參與者在相近時段出現噁心、嘔吐與腹瀉的急報。調查不能一開始就問哪道菜有問題，而要先把所有參與者、可判定者與無法判定者分開。",
        "narrative": "這一幕把急報轉成可分析的事件分母。活動名冊提供 312 名參與者，食品暴露表記錄每個人的回答，病例函式則把《金玉良炎疫情操作手冊》的文字規則轉成可重跑欄位；因此後面每一次比較都有清楚的分母來源。",
        "question": "312 名參與者中，誰有足夠資料可判定？不同病例定義會辨認多少病例？",
        "finding": f"在 {int(primary_row.classifiable_n)} 名可依 {CASE_DEFINITION} 定義判定的參與者中，{int(primary_row.cases)} 人符合主要病例，侵襲率為 {primary_row.attack_rate:.1%}。同時保留 sensitive 與 strict 結果，讓病例規則的寬嚴不會被藏在單一數字裡。",
        "next": "病例名單已固定後，下一步把這些病例放回人物角色與三處宮苑，確認急報是局部集中，還是跨場地共享來源的事件。",
    },
    {
        "stage": "A2", "title": "描述 Person／Place", "chart_id": "person_place_attack_rates",
        "pipeline": "病例名單 → 依角色與場地分組 → 各組建立可分類分母 → 比較侵襲率",
        "cause": "病例名單固定後，御醫署先問：這是不是某一類宮人或某一座宮苑的局部問題？如果三處宮苑都出現病例，調查焦點就不能只停留在單一房舍或單一職務。",
        "narrative": "左側 Place 面板把病例放回承月臺、蘅芳苑與昭華殿，右側 Person 面板則比較五類角色。每個群組都使用自己的可分類者作分母，讓「某組病例比例較高」不會被誤讀成「某組人數最多」。",
        "question": "各群組以自己的可分類者為分母時，病例集中在哪裡？",
        "finding": "三處宮苑均有病例，且不同角色群組的侵襲率並不完全相同；這種跨場地、跨角色的分布較像共同用餐或共同暴露，需要把調查推向菜單、食品暴露與供膳流程。",
        "next": "下一步檢查症狀開始時間，確認病例是否在宴後相近時段形成集中波峰，並判斷事件較接近單次共同來源還是持續傳播。",
    },
    {
        "stage": "A3", "title": "檢視流行曲線", "chart_id": "epidemic_curve",
        "pipeline": "病例與時間欄位 → 以宴後症狀開始時間分箱 → 比較波峰與共同暴露時序",
        "cause": "跨場地病例提高共同來源假說，但場地分布本身不能告訴我們暴露發生在何時；因此需要把每名病例的症狀開始時間放回宴席後的時間軸。",
        "narrative": "流行曲線把一串急報變成時間上的形狀。出版版使用講師指定的分箱，柱頂直接標出每個時間箱的新發病例數；互動版則保留不同分箱，方便檢查波峰是否只是過度粗化或過度切細的視覺效果。",
        "question": f"使用 {PUBLICATION_EPI_BIN_HOURS} 小時分箱時，{int(primary_row.cases)} 名病例呈現什麼時間型態？",
        "finding": "病例在宴後相近時段形成集中波峰，與一次共同來源暴露相容；不同分箱的病例總數仍一致，因此後續可以把時間型態與菜單及食品暴露接起來。",
        "next": "時間線支持共同來源後，下一步依菜單與食品問卷逐項比較食用者與未食用者，尋找最值得追查的食品候選。",
    },
    {
        "stage": "A4", "title": "完成食品粗分析", "chart_id": "food_crude_forest",
        "pipeline": "食品暴露表 → 各食品 2×2 表 → OR、95% CI 與 Fisher exact test → 建立候選排序",
        "cause": "時間型態與單次共同來源相容後，食品問卷成為主要線索。御醫署先不急著宣告媒介，而是逐道菜比較食用者與未食用者的病例差異，並保留每道食品自己的完整案例分母。",
        "narrative": "森林圖讓十項食品同時站在同一條 OR 軸上。翡翠凝露羹的食用者與未食用者差異最大，合歡桂露飲也呈現關聯；但這只是把兩項菜色帶進下一輪盤問，還沒有回答兩者是否被同一批人一起食用。",
        "question": "哪些食品的食用者與未食用者呈現最大的發病差異？",
        "finding": f"{food_label_map[candidate_1]}粗 OR {jade.odds_ratio:.2f}（95% CI {jade.ci_low:.2f}–{jade.ci_high:.2f}），{food_label_map[candidate_2]}粗 OR {osmanthus.odds_ratio:.2f}。粗分析把兩項食品列為候選，但不能排除共食與場地混雜。",
        "next": "下一步檢查兩項候選是否經常由同一批人一起食用；如果高度共食，兩個粗 OR 可能是在描述同一條供膳路徑。",
    },
    {
        "stage": "A5", "title": "辨認食品共食結構", "chart_id": "food_coexposure_heatmap",
        "pipeline": "暴露長表 → 人 × 食品矩陣 → 成對 Pearson 相關 → 找出候選共食與混雜結構",
        "cause": "粗分析同時指向兩項食品，調查需要回到宴席座次與供膳安排：這兩項食品是否都出現在同一批人的餐盤裡？",
        "narrative": "Heatmap 不是在尋找一個神奇的紅色格子，而是在重建宴席上的共食結構。若兩項食品被同一批人一起食用，粗分析看到的病例差異可能屬於整套餐點或同一場地，而不是其中一道菜獨立造成。",
        "question": "食品暴露之間是否存在足以造成混雜的相關結構？",
        "finding": f"兩項候選食品的暴露相關係數為 {corr.loc[candidate_1, candidate_2]:.2f}，顯示它們確實有共食結構；因此粗 OR 必須透過場地分層與多變項模型重新拆解。",
        "next": "下一步把兩項候選與場地放進同一分析架構，觀察主要候選的訊號在控制共食與場地後是否仍然保留。",
    },
    {
        "stage": "A6", "title": "比較粗、分層與調整效果", "chart_id": "adjusted_effect_comparison",
        "pipeline": "候選食品與場地 → Mantel–Haenszel 分層 → logistic regression → 比較調整前後 OR",
        "cause": "共食結構使兩項食品的粗 OR 不能直接被當成兩個獨立證據。調查接著以場地分層，並把兩項候選與指定調整因子放進同一個模型。",
        "narrative": "這一幕是故事中的轉折：問題從「哪道菜看起來最可疑」變成「在同時考慮另一道菜與宮苑後，哪個訊號仍能解釋病例差異」。模型不是替調查做最後宣判，而是檢查候選食品是否能經過混雜結構的考驗。",
        "question": "控制另一項候選與場地後，哪個食品訊號仍然保留？",
        "finding": f"{food_label_map[candidate_1]}調整 OR {adj.loc[candidate_1, 'odds_ratio']:.2f}（95% CI {adj.loc[candidate_1, 'ci_low']:.2f}–{adj.loc[candidate_1, 'ci_high']:.2f}），而 {food_label_map[candidate_2]} 的調整信賴區間包含 1。模型使用 {diagnostics['n_observations']} 筆完整案例並成功收斂，主要候選訊號沒有被共食與場地完全解釋掉。",
        "next": "下一步檢查這個主要候選是否依賴少數未知暴露的編碼；若在合理的極端假設下方向仍一致，故事才有足夠穩健性進入直接證據階段。",
    },
    {
        "stage": "A7", "title": "執行暴露缺失敏感度分析", "chart_id": "missing_exposure_sensitivity",
        "pipeline": "固定病例分類 → 對主要候選建立三種缺失編碼 → 重新計算 OR 與 95% CI → 比較方向與量級",
        "cause": "主要候選已浮現，但食品問卷仍有少量 unknown。調查不能把未知回答偷偷塞進某一組，因此用完整案例、未知全視為未食用、未知全視為食用三種情境揭露假設。",
        "narrative": "這一幕像是對主要線索的壓力測試：同一批病例不變，只改變未知暴露的處理方式。如果必須把未知者全部安排成食用或全部安排成未食用，結論才會翻轉，那麼前面的訊號就不夠穩定；如果方向維持，才適合帶著不確定性進入檢驗證據。",
        "question": "將未知全視為食用或未食用時，主要效果方向是否改變？",
        "finding": "三個情境的 OR 都位於 1 的同一側，主要方向在兩個極端暴露假設下維持一致；這支持主要候選的訊號不是由單一缺失值編碼製造出來的。",
        "next": "下一步從問卷關聯轉向人體、食品分型與候膳環境紀錄，確認統計線索是否能與直接證據及可疑保存條件接合。",
    },
    {
        "stage": "A8", "title": "整合檢驗與環境證據", "chart_id": "lab_sampling_summary", "extra_chart_id": "environment_triangulation",
        "pipeline": "Release 2 人體與食品檢驗 → Release 3 環境紀錄 → 比對 YL-4、來源與候膳條件 → 建立證據層級",
        "cause": "觀察性分析已把翡翠凝露羹推到主要候選位置，現在需要問更接近事件現場的問題：病例檢體、食品檢體與候膳環境，是否能指向同一條來源鏈？",
        "narrative": "人體檢驗圖先交代陽性結果出現在哪些時間窗與檢驗方法；環境三角圖再把各宮苑的暴露侵襲率、候膳溫度與保留時間放在一起。這兩張圖不是兩個獨立結論，而是用來檢查統計候選是否有直接檢體與合理環境條件支撐。",
        "question": "人體檢驗、食品分型與候膳環境是否指向一致解釋？",
        "finding": "人體病例檢體與翡翠凝露羹食品檢體出現相符 YL-4；場地候膳條件與暴露者侵襲率也提供方向一致的支持。Petalomyces innocua 依手冊屬背景環境發現，因此不能讓它取代主要候選的證據鏈。",
        "next": "以證據層級形成結案摘要：群聚與病例定義回答事件是否存在，人體與食品分型回答疾病及媒介是否相符，環境圖則說明哪一段供膳或保存條件值得改善與追查。",
    },
]
EXECUTIVE_SUMMARY = (
    f"本案始於千秋宴後集中出現的嘔吐與腹瀉急報。御醫署先固定 312 名參與者的事件分母，再以 {CASE_DEFINITION} 病例定義判定 {int(primary_row.classifiable_n)} 人，辨認 {int(primary_row.cases)} 名主要病例。"
    f"病例跨三處宮苑分布，且在宴後相近時段形成波峰；食品粗分析把{food_label_map[candidate_1]}與{food_label_map[candidate_2]}帶入候選範圍。"
    f"在檢查共食結構、場地分層與多變項模型後，{food_label_map[candidate_1]}調整 OR 為 {adj.loc[candidate_1, 'odds_ratio']:.2f}，{food_label_map[candidate_2]}的調整信賴區間包含 1。"
    "缺失暴露的極端情境沒有改變主要方向；後續人體、食品與環境資料則提供與主要候選一致的直接與脈絡證據。"
)
PIPELINE_SUMMARY = "急報與固定資料 → 病例定義與分母 → Person／Place／Time → 食品粗分析 → 共食結構 → 場地分層與多變項調整 → 缺失值敏感度 → 人體／食品／環境三角驗證 → 圖表、Word、PDF 與互動 HTML 交付。"
CLOSING_SUMMARY = (
    f"本案從千秋宴後的急報開始，先用固定分母與透明病例定義確認事件範圍，再依序完成 Person／Place／Time、食品粗分析、共食檢查、場地分層、多變項調整與缺失值敏感度。"
    f"目前證據鏈共同支持{food_label_map[candidate_1]}為主要共同來源媒介候選；{food_label_map[candidate_2]}的粗關聯較可能受到共食與場地結構影響。"
    "人體、食品分型與候膳環境資料讓統計線索與直接證據互相對照，但結案仍應保留證據層級：群聚成立、疾病身分、食品媒介與環境改善是不同問題。"
)
REPORT_METHODS = (
    f"本案採回溯性群聚調查架構。研究母群為千秋宴活動名冊中的 312 名參與者；病例依《金玉良炎疫情操作手冊》轉譯為 primary、sensitive 與 strict 三種可重跑定義，主要分析預設採用 {CASE_DEFINITION}。"
    "資料來源包括活動名冊與症狀問卷、食品暴露問卷、菜單與供膳紀錄、人體與食品檢驗結果，以及候膳環境紀錄。分析先完成 Person、Place、Time 描述，再以食品 2×2 表與粗 OR 產生假說，接著檢查共食相關、場地分層與多變項 logistic regression，最後以缺失暴露敏感度和人體／食品／環境證據檢查解釋的一致性。"
)
REPORT_DISCUSSION = (
    "整體結果支持翡翠凝露羹是本次事件的主要共同來源媒介候選，但這個判斷來自多層證據的匯合，而不是單一顯著檢定。時間上的集中波峰說明共同來源型態，食品粗分析提供候選，調整後訊號與缺失敏感度支持關聯具有一定穩健性，YL-4 在病例與食品檢體中的相符則把統計線索連到直接證據。"
    "本案仍有幾項限制：食品暴露依賴事後問卷，部分回答缺失；環境與食品採樣只代表採樣時點和少數樣本；背景環境發現不能自動等同於致病媒介；資料沒有完整追溯所有食材與供應鏈。因此報告應使用「支持」「相符」「主要候選」等證據語言，而不是宣稱單一分析已證明全部污染環節。"
)
REPORT_ACTIONS = (
    "在故事中的處置上，御醫署應先保全剩餘食品、菜單、供膳與候膳紀錄，停止可疑食品繼續供應，追查翡翠凝露羹的製備、保存與分送路徑；同時依採檢時機補強病例與食品檢體的分型比對，針對候膳溫度、保留時間、器具清潔與交叉污染風險完成環境矯正。這些行動不是統計結果的附錄，而是把調查發現轉成降低下一場宴席風險的措施。"
)

display(pd.DataFrame(REPORT_SECTIONS)[["stage", "title", "pipeline", "question"]])

display(pd.DataFrame(REPORT_SECTIONS)[["stage", "title", "question", "next"]])


In [ ]:
# ─────────────────────────────────────────────────────────────
# 11.2｜輸出 Word 與列印版 PDF
# Word 使用 standard business brief 骨架，加上宮廷色票；PDF 使用同一內容與出版 PNG。
# ─────────────────────────────────────────────────────────────
from docx import Document
from docx.enum.section import WD_SECTION
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_CELL_VERTICAL_ALIGNMENT
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Cm, Inches, Pt, RGBColor
from weasyprint import HTML

def set_cell_shading(cell, fill):
    tc_pr = cell._tc.get_or_add_tcPr()
    shading = OxmlElement("w:shd")
    shading.set(qn("w:fill"), fill.replace("#", ""))
    tc_pr.append(shading)

def set_run_font(run, size=None, bold=None, color=None):
    run.font.name = "Noto Sans CJK TC"
    run._element.get_or_add_rPr().rFonts.set(qn("w:eastAsia"), "Noto Sans CJK TC")
    if size is not None:
        run.font.size = Pt(size)
    if bold is not None:
        run.bold = bold
    if color is not None:
        run.font.color.rgb = RGBColor.from_string(color.replace("#", ""))

def add_label_box(document, label, text, fill="F4EDDF"):
    table = document.add_table(rows=1, cols=2)
    table.autofit = False
    table.columns[0].width = Cm(2.7)
    table.columns[1].width = Cm(14.7)
    left, right = table.rows[0].cells
    set_cell_shading(left, fill)
    set_cell_shading(right, fill)
    left.vertical_alignment = right.vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER
    lp = left.paragraphs[0]; rp = right.paragraphs[0]
    lr = lp.add_run(label); rr = rp.add_run(text)
    set_run_font(lr, 9, True, COLORS["cinnabar"])
    set_run_font(rr, 9.5, False, COLORS["ink"])
    document.add_paragraph().paragraph_format.space_after = Pt(0)

def build_word_report(output_path):
    document = Document()
    section = document.sections[0]
    section.page_width = Inches(8.27); section.page_height = Inches(11.69)
    section.top_margin = Cm(1.8); section.bottom_margin = Cm(1.7)
    section.left_margin = Cm(1.9); section.right_margin = Cm(1.9)

    styles = document.styles
    for style_name in ["Normal", "Title", "Heading 1", "Heading 2"]:
        style = styles[style_name]
        style.font.name = "Noto Sans CJK TC"
        style._element.rPr.rFonts.set(qn("w:eastAsia"), "Noto Sans CJK TC")
    styles["Normal"].font.size = Pt(10)
    styles["Heading 1"].font.size = Pt(19); styles["Heading 1"].font.color.rgb = RGBColor(46, 98, 85)

    header = section.header.paragraphs[0]
    header.text = "御醫署群聚調查奏報｜完全虛構教學案例"
    set_run_font(header.runs[0], 8, True, COLORS["jade"])
    footer = section.footer.paragraphs[0]
    footer.alignment = WD_ALIGN_PARAGRAPH.CENTER
    footer.text = "《金玉良炎》公衛資料分析與 AI Code 協作工作坊"
    set_run_font(footer.runs[0], 8, False, COLORS["muted"])

    title = document.add_paragraph()
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER
    title.paragraph_format.space_before = Pt(80)
    run = title.add_run("千秋宴金玉良炎\n群聚調查報告")
    set_run_font(run, 27, True, COLORS["jade"])
    subtitle = document.add_paragraph()
    subtitle.alignment = WD_ALIGN_PARAGRAPH.CENTER
    sr = subtitle.add_run("PUBLICATION MODE｜講師示範")
    set_run_font(sr, 11, True, COLORS["gold"])
    document.add_paragraph()
    fiction = document.add_paragraph()
    fiction.alignment = WD_ALIGN_PARAGRAPH.CENTER
    fr = fiction.add_run("疾病、人物、場所、檢驗、事件與數值均為完全虛構的合成教學內容")
    set_run_font(fr, 9, False, COLORS["cinnabar"])
    document.add_page_break()

    document.add_heading("執行摘要", level=1)
    summary = document.add_paragraph(EXECUTIVE_SUMMARY)
    for run in summary.runs:
        set_run_font(run, 10.5)
    document.add_heading("分析路徑", level=2)
    pipeline = document.add_paragraph(PIPELINE_SUMMARY)
    for run in pipeline.runs:
        set_run_font(run, 10.2, False, COLORS["jade"])
    document.add_heading("核心發現", level=2)
    core = document.add_paragraph(f"{food_label_map[candidate_1]}粗 OR {jade.odds_ratio:.2f}，調整 OR {adj.loc[candidate_1, 'odds_ratio']:.2f}；這是整條證據鏈中的主要統計訊號。")
    for run in core.runs:
        set_run_font(run, 10.2)
    principle = document.add_paragraph("判讀時將時間、人物、地點、食品暴露、檢驗與環境條件放在同一條證據鏈上；任何單一 OR、CI、p 值或陽性檢驗都不獨立代表完整結論。")
    for run in principle.runs:
        set_run_font(run, 10.2, False, COLORS["muted"])

    document.add_heading("調查方法與證據框架", level=1)
    methods = document.add_paragraph(REPORT_METHODS)
    for run in methods.runs:
        set_run_font(run, 10.2)

    for index, item in enumerate(REPORT_SECTIONS):
        document.add_heading(f"{item['stage']}｜{item['title']}", level=1)

        lead = document.add_paragraph(f"{item['cause']} {item['narrative']}")
        for run in lead.runs:
            set_run_font(run, 10.5)
        document.add_heading("調查問題", level=2)
        question = document.add_paragraph(item["question"])
        for run in question.runs:
            set_run_font(run, 10.2, False, COLORS["jade"])
        document.add_heading("分析方法與流程", level=2)
        method = document.add_paragraph(item["pipeline"])
        for run in method.runs:
            set_run_font(run, 10.0, False, COLORS["muted"])

        # 圖表嵌入結果段落中，不再用欄位卡片切碎閱讀節奏。
        image_width = Cm(12.2) if item.get("extra_chart_id") else Cm(16.8)
        if item.get("chart_id"):
            image_path = PUBLICATION_DIR / f"{item['chart_id']}.png"
            document.add_picture(str(image_path), width=image_width)
            document.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER
        if item.get("extra_chart_id"):
            image_path = PUBLICATION_DIR / f"{item['extra_chart_id']}.png"
            document.add_picture(str(image_path), width=image_width)
            document.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER

        document.add_heading("結果與判讀", level=2)
        finding = document.add_paragraph(item["finding"])
        for run in finding.runs:
            set_run_font(run, 10.5)
        document.add_heading("下一步調查", level=2)
        next_step = document.add_paragraph(item["next"])
        for run in next_step.runs:
            set_run_font(run, 10.2, False, COLORS["jade"])

    document.add_page_break()
    document.add_heading("討論、限制與公共衛生意涵", level=1)
    discussion = document.add_paragraph(REPORT_DISCUSSION)
    for run in discussion.runs:
        set_run_font(run, 10.2)
    document.add_heading("公共衛生行動", level=2)
    actions = document.add_paragraph(REPORT_ACTIONS)
    for run in actions.runs:
        set_run_font(run, 10.2)

    document.add_page_break()
    document.add_heading("結案摘要", level=1)
    conclusion = document.add_paragraph(CLOSING_SUMMARY)
    for run in conclusion.runs:
        set_run_font(run, 10.5)
    document.save(output_path)
    return Path(output_path)

def image_data_uri(path):
    encoded = base64.b64encode(Path(path).read_bytes()).decode("ascii")
    return f"data:image/png;base64,{encoded}"

def build_print_html():
    section_html = []
    for item in REPORT_SECTIONS:
        images = ""
        for key in [item.get("chart_id"), item.get("extra_chart_id")]:
            if key:
                images += f'<img src="{image_data_uri(PUBLICATION_DIR / (key + ".png"))}" alt="{key}">'
        stage_class = "stage two-chart" if item.get("extra_chart_id") else "stage"
        section_html.append(f'''<article class="{stage_class}">
          <div class="stage-code">{item["stage"]}</div><h2>{item["title"]}</h2>
          <p class="lead">{item["cause"]} {item["narrative"]}</p>
          <h3>調查問題</h3><p class="question-text">{item["question"]}</p>
          <h3>分析方法與流程</h3><p class="method-text">{item["pipeline"]}</p>
          {images}
          <h3>結果與判讀</h3><p>{item["finding"]}</p>
          <h3>下一步調查</h3><p class="next-text">{item["next"]}</p>
        </article>''')
    return f'''<!doctype html><html lang="zh-Hant"><head><meta charset="utf-8"><style>
    @page {{ size:A4; margin:16mm 16mm 17mm; @bottom-center {{ content:"《金玉良炎》講師示範 · " counter(page); color:#746D63; font-size:8pt; }} }}
    * {{ box-sizing:border-box }} body {{ font-family:"Noto Sans CJK TC","Noto Sans CJK",sans-serif; color:#25352F; font-size:10.5pt; line-height:1.85; margin:0 }}
    .cover {{ height:252mm; background:#2E6255; color:#FFF8E8; border-top:8mm solid #B38A3E; padding:55mm 20mm 20mm; page-break-after:always }}
    .cover small {{ color:#E4C47D; letter-spacing:.16em; font-weight:800 }} .cover h1 {{ font-size:31pt; line-height:1.25; margin:8mm 0 }}
    .fiction {{ border:1px solid #D58B76; padding:3mm; color:#FFD8CD }}
    .summary {{ page-break-before:always }} .stage {{ margin:14mm 0 18mm }} h2 {{ color:#2E6255; font-size:20pt; margin:0 0 5mm }}
    .stage-code {{ color:#934631; border:1px solid #934631; display:inline-block; padding:2mm 3mm; font-weight:800; margin-bottom:3mm }}
    .stage h3,.summary h3 {{ color:#934631; font-size:11pt; margin:6mm 0 1.5mm; border-bottom:1px solid #D9CDBA; padding-bottom:1mm }}
    .lead {{ font-size:11.5pt; line-height:1.95; margin:0 0 5mm }} .question-text {{ color:#2E6255; font-style:italic; margin-top:0 }} .method-text {{ color:#746D63; font-size:9.8pt; margin-top:0 }} .next-text {{ color:#2E6255 }}
    img {{ width:100%; max-height:112mm; object-fit:contain; margin:5mm 0 }} .two-chart img {{ max-height:58mm; margin:2mm 0 }}
    </style></head><body>
    <section class="cover"><small>御醫署群聚調查奏報｜PUBLICATION MODE</small><h1>千秋宴金玉良炎<br>群聚調查報告</h1><p>這是一份以疫情調查報告結構撰寫的合成教學案例：從急報與病例定義出發，經過描述性與分析性流行病學，再以人體、食品與環境證據形成結案判斷。</p><p class="fiction">疾病、人物、場所、檢驗、事件與數值均為完全虛構的合成教學內容。</p></section>
    <section class="summary"><h2>執行摘要</h2><p class="lead">{EXECUTIVE_SUMMARY}</p><h3>分析路徑</h3><p class="method-text">{PIPELINE_SUMMARY}</p></section>
    <section class="summary"><h2>調查方法與證據框架</h2><p>{REPORT_METHODS}</p></section>
    {''.join(section_html)}
    <section class="summary"><h2>討論、限制與公共衛生意涵</h2><p>{REPORT_DISCUSSION}</p><h3>公共衛生行動</h3><p>{REPORT_ACTIONS}</p></section>
    <section class="summary"><h2>結案摘要</h2><p>{CLOSING_SUMMARY}</p><h3>判讀界線</h3><p>這項結論來自多種證據的一致性；單一圖、OR、CI、p 值或陽性檢驗各自只回答證據鏈中的一部分。</p></section>
    </body></html>'''

word_path = OUTPUT_ROOT / "jinyuliang_investigation_report.docx"
pdf_path = OUTPUT_ROOT / "jinyuliang_investigation_report_print.pdf"
print_html_path = OUTPUT_ROOT / "jinyuliang_investigation_report_print.html"

if BUILD_WORD:
    build_word_report(word_path)
    print("Word：", word_path.resolve())
if BUILD_PDF:
    print_html = build_print_html()
    print_html_path.write_text(print_html, encoding="utf-8")
    HTML(string=print_html, base_url=str(Path.cwd())).write_pdf(pdf_path)
    print("列印版 PDF：", pdf_path.resolve())


## 12｜互動 HTML 延伸

這一段才把前面已經核對過的分析表轉成 Plotly。互動圖與出版圖共用同一批計算結果；hover 用來探索細節，右上角可下載 SVG。

### 報告閱讀提示

互動版適合探索細節，例如切換流行曲線分箱、查看單一食品的 hover 數值或下載 SVG；出版版則適合固定版面、列印與審閱。兩者若出現視覺差異，應回到前面的結果表與 `chart_index.csv` 核對，而不是把互動效果當成新的分析。


In [ ]:
# ─────────────────────────────────────────────────────────────
# 12.1｜從既有結果表建立互動 Plotly 圖
# 計算不在這裡重做，避免出版圖與互動圖出現不同答案。
# ─────────────────────────────────────────────────────────────
def register_interactive(key, fig, title, height=520, x_title=None, y_title=None):
    apply_plotly_theme(fig, title, height, x_title, y_title)
    INTERACTIVE_FIGURES[key] = fig
    return fig

# Person／Place
fig = make_subplots(rows=1, cols=2, subplot_titles=("Place｜三處宮苑", "Person｜角色群組"), horizontal_spacing=0.17)
for column, frame, color in [(1, site_rates.sort_values("attack_rate"), COLORS["jade"]), (2, role_rates.sort_values("attack_rate"), COLORS["cinnabar"])]:
    fig.add_trace(go.Bar(
        x=frame["attack_rate"], y=frame["label"], orientation="h", marker_color=color,
        text=frame.apply(lambda r: f"{int(r.cases)}/{int(r.classifiable_n)} · {r.attack_rate:.1%}", axis=1),
        textposition="outside", customdata=np.column_stack([frame["cases"], frame["classifiable_n"]]),
        hovertemplate="<b>%{y}</b><br>侵襲率 %{x:.1%}<br>病例／分母 %{customdata[0]:.0f}／%{customdata[1]:.0f}<extra></extra>",
        showlegend=False,
    ), row=1, col=column)
register_interactive("person_place_attack_rates", fig, "主要病例的 Person／Place 分布", 510, "侵襲率", None)
fig.update_xaxes(tickformat=".0%")

# 可切換分箱的流行曲線
fig = go.Figure()
for index, hours in enumerate(INTERACTIVE_EPI_BINS):
    frame = epi_tables[hours]
    fig.add_trace(go.Bar(
        x=frame["bin_start"], y=frame["cases"], width=hours*60*60*1000*0.92,
        marker_color=COLORS["cinnabar"], visible=(hours == PUBLICATION_EPI_BIN_HOURS),
        customdata=np.column_stack([frame["bin_end"].dt.strftime("%m-%d %H:%M")]),
        hovertemplate="<b>%{x|%m-%d %H:%M} 至 %{customdata[0]}</b><br>新發病例 %{y:.0f}<extra></extra>",
        name=f"{hours} 小時",
    ))
buttons = []
for index, hours in enumerate(INTERACTIVE_EPI_BINS):
    buttons.append({"label": f"{hours} 小時", "method": "update", "args": [
        {"visible": [position == index for position in range(len(INTERACTIVE_EPI_BINS))]},
        {"title.text": f"主要病例流行曲線｜{hours} 小時分箱"},
    ]})
register_interactive("epidemic_curve", fig, f"主要病例流行曲線｜{PUBLICATION_EPI_BIN_HOURS} 小時分箱", 520, "發病日期時間", "新發病例數")
fig.update_layout(updatemenus=[{"buttons": buttons, "x": 1, "xanchor": "right", "y": 1.18}], bargap=0.04)
fig.update_xaxes(tickformat="%m-%d<br>%H:%M", nticks=9)

# 食品粗 OR
frame = food_results.sort_values("odds_ratio")
fig = go.Figure(go.Scatter(
    x=frame["odds_ratio"], y=frame["food_name_zh"], mode="markers",
    marker={"size": 11, "color": COLORS["cinnabar"]},
    error_x={"type":"data", "symmetric":False, "array":frame["ci_high"]-frame["odds_ratio"], "arrayminus":frame["odds_ratio"]-frame["ci_low"]},
    customdata=np.column_stack([frame["ci_low"], frame["ci_high"], frame["n_complete"], frame["a"], frame["b"], frame["c"], frame["d"]]),
    hovertemplate="<b>%{y}</b><br>粗 OR %{x:.2f}<br>95% CI %{customdata[0]:.2f}–%{customdata[1]:.2f}<br>完整分母 %{customdata[2]:.0f}<br>2×2 a=%{customdata[3]:.0f}, b=%{customdata[4]:.0f}, c=%{customdata[5]:.0f}, d=%{customdata[6]:.0f}<extra></extra>",
))
fig.add_vline(x=1, line_dash="dash", line_color=COLORS["ink"])
register_interactive("food_crude_forest", fig, "十項食品的粗 OR 與 95% CI", 620, "粗勝算比（log scale）", None)
fig.update_xaxes(type="log")

# 共食 heatmap
fig = go.Figure(go.Heatmap(z=matrix, x=ordered_labels, y=ordered_labels, zmin=-1, zmax=1, zmid=0,
                           colorscale="RdBu_r", text=np.round(matrix,2), texttemplate="%{text:.2f}",
                           hovertemplate="<b>%{y}</b> × <b>%{x}</b><br>相關係數 %{z:.2f}<extra></extra>"))
register_interactive("food_coexposure_heatmap", fig, "十項食品的共食相關性", 700)
fig.update_xaxes(tickangle=-32); fig.update_yaxes(autorange="reversed")

# 調整效果
frame = effect_plot.iloc[::-1]
fig = go.Figure(go.Scatter(
    x=frame["odds_ratio"], y=frame["label"], mode="markers",
    marker={"size":12, "color":[COLORS["blue"] if food_label_map[candidate_2] in x else COLORS["cinnabar"] for x in frame["label"]]},
    error_x={"type":"data", "symmetric":False, "array":frame["ci_high"]-frame["odds_ratio"], "arrayminus":frame["odds_ratio"]-frame["ci_low"]},
    customdata=np.column_stack([frame["ci_low"], frame["ci_high"]]),
    hovertemplate="<b>%{y}</b><br>OR %{x:.2f}<br>95% CI %{customdata[0]:.2f}–%{customdata[1]:.2f}<extra></extra>",
))
fig.add_vline(x=1, line_dash="dash", line_color=COLORS["ink"])
register_interactive("adjusted_effect_comparison", fig, "候選食品的粗、分層與調整效果", 500, "勝算比（log scale）", None)
fig.update_xaxes(type="log")

# 缺失敏感度
frame = missing_sensitivity.iloc[::-1]
fig = go.Figure(go.Scatter(
    x=frame["odds_ratio"], y=frame["label"], mode="markers+text", text=frame["odds_ratio"].map(lambda x:f"OR {x:.2f}"), textposition="top center",
    marker={"size":12, "color":COLORS["jade"]},
    error_x={"type":"data", "symmetric":False, "array":frame["ci_high"]-frame["odds_ratio"], "arrayminus":frame["odds_ratio"]-frame["ci_low"]},
    customdata=np.column_stack([frame["ci_low"], frame["ci_high"], frame["n_complete"]]),
    hovertemplate="<b>%{y}</b><br>OR %{x:.2f}<br>95% CI %{customdata[0]:.2f}–%{customdata[1]:.2f}<br>分析分母 %{customdata[2]:.0f}<extra></extra>",
))
fig.add_vline(x=1, line_dash="dash", line_color=COLORS["ink"])
register_interactive("missing_exposure_sensitivity", fig, f"{food_label_map[candidate_1]}：暴露缺失敏感度分析", 440, "勝算比（log scale）", None)
fig.update_xaxes(type="log")

# 人體檢驗
fig = px.bar(lab_summary, x="sampling_window", y="positive_fraction", color="test_name", barmode="group",
             custom_data=["positives", "tests", "persons"], color_discrete_sequence=[COLORS["cinnabar"], COLORS["jade"], COLORS["blue"]])
fig.update_traces(hovertemplate="<b>%{fullData.name}</b><br>採檢時間 %{x}<br>陽性比例 %{y:.1%}<br>陽性／檢驗 %{customdata[0]:.0f}／%{customdata[1]:.0f}<br>受檢人數 %{customdata[2]:.0f}<extra></extra>")
register_interactive("lab_sampling_summary", fig, "人體檢驗陽性比例與採檢時機", 510, "相對發病時間", "陽性比例")
fig.update_yaxes(tickformat=".0%", range=[0,1.08])

# 環境三角驗證
fig = px.scatter(triangle, x="median_temperature_c", y="attack_rate", size="holding_minutes", color="site_name_zh", text="site_name_zh",
                 custom_data=["exposed_n", "cases", "holding_minutes", "records"],
                 color_discrete_sequence=[COLORS["cinnabar"], COLORS["gold"], COLORS["jade"]], size_max=44)
fig.update_traces(textposition="top center", hovertemplate="<b>%{fullData.name}</b><br>候膳溫度 %{x:.1f}°C<br>侵襲率 %{y:.1%}<br>病例／暴露分母 %{customdata[1]:.0f}／%{customdata[0]:.0f}<br>候膳時間 %{customdata[2]:.0f} 分<extra></extra>")
register_interactive("environment_triangulation", fig, f"{food_label_map[candidate_1]}：場地環境三角驗證", 530, "候膳溫度中位數（°C）", "暴露者侵襲率")
fig.update_yaxes(tickformat=".0%")

print("已建立互動圖：", list(INTERACTIVE_FIGURES))


In [ ]:
# ─────────────────────────────────────────────────────────────
# 12.2｜輸出單一、離線、具有前因後果的互動 HTML
# ─────────────────────────────────────────────────────────────
def plotly_fragment(key, include_plotlyjs=False):
    return pio.to_html(
        INTERACTIVE_FIGURES[key], full_html=False,
        include_plotlyjs=True if include_plotlyjs else False,
        config=svg_config(key), default_width="100%", default_height="560px",
    )

interactive_sections = []
first_plot = True
for item in REPORT_SECTIONS:
    charts = ""
    for key in [item.get("chart_id"), item.get("extra_chart_id")]:
        if key:
            charts += plotly_fragment(key, include_plotlyjs=first_plot)
            first_plot = False
    interactive_sections.append(f'''<article class="stage">
      <header><span>{item["stage"]}</span><h2>{item["title"]}</h2></header>
      <p class="lead">{item["cause"]} {item["narrative"]}</p>
      <h3>調查問題</h3><p class="question-text">{item["question"]}</p>
      <h3>分析方法與流程</h3><p class="method-text">{item["pipeline"]}</p>
      <div class="evidence">{charts if charts else '<p class="no-chart">本幕先建立病例與分母，沒有獨立圖表。</p>'}</div>
      <h3>結果與判讀</h3><p>{item["finding"]}</p>
      <h3>下一步調查</h3><p class="next-text">{item["next"]}</p>
    </article>''')

interactive_html = f'''<!doctype html><html lang="zh-Hant"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>金玉良炎互動調查報告</title><style>
:root{{--paper:#F2EBDD;--card:#FFFDF8;--ink:#25352F;--jade:#2E6255;--red:#934631;--gold:#B38A3E;--line:#D9CDBA}}
*{{box-sizing:border-box}} body{{margin:0;background:var(--paper);color:var(--ink);font-family:"Noto Sans TC","Microsoft JhengHei",sans-serif;line-height:1.75}}
.hero{{padding:68px max(28px,calc((100% - 1100px)/2));background:var(--jade);color:#FFF8E8;border-bottom:8px solid var(--gold)}} .hero h1{{font-size:clamp(36px,6vw,62px);line-height:1.2;margin:.25em 0}}
.hero small{{color:#E4C47D;letter-spacing:.14em;font-weight:800}} main{{max-width:1100px;margin:auto;padding:48px 22px 90px}}
.stage{{margin:0 0 54px;padding:32px;background:var(--card);border:1px solid var(--line);box-shadow:0 12px 32px #4A3B2512}} .stage header{{display:flex;gap:16px;align-items:center;border-bottom:1px solid var(--line);padding-bottom:17px}}
.stage header span{{width:56px;height:56px;display:grid;place-items:center;border:1px solid var(--red);color:var(--red);font-weight:800}} h2{{color:var(--jade);font-size:27px;margin:0}}
.stage h3{{color:var(--red);font-size:13px;margin:22px 0 5px;border-bottom:1px solid var(--line);padding-bottom:5px}} .lead{{font-size:17px;line-height:1.9}} .question-text{{color:var(--jade);font-style:italic}} .method-text{{color:#746D63;font-size:14px}} .next-text{{color:var(--jade)}} .evidence{{margin:28px 0}} .no-chart{{color:#746D63;font-style:italic}}
.fiction{{display:inline-block;border:1px solid #D58B76;padding:4px 10px;color:#FFD8CD}} footer{{padding:30px;text-align:center;color:#746D63;font-size:12px}}
</style></head><body><header class="hero"><p class="fiction">完全虛構的公共衛生教學案例</p><small>互動延伸版｜HOVER · ZOOM · SVG</small><h1>千秋宴金玉良炎群聚調查</h1><p>{EXECUTIVE_SUMMARY}</p><p>互動圖與出版圖共用同一批計算結果；出版報告負責直接標值，這一版負責延伸探索。</p></header><main><section class="stage"><header><span>方法</span><h2>調查方法與證據框架</h2></header><div class="logic cause"><b>調查設計</b><p>{REPORT_METHODS}</p></div><div class="logic next"><b>證據整合原則</b><p>先用描述性流行病學定位時間、人物與地點，再用分析性流行病學形成候選，最後以實驗室、食品與環境資料交叉檢查；任何單一證據都不取代整條證據鏈。</p></div></section>{''.join(interactive_sections)}</main><section class="stage"><header><span>結案</span><h2>討論與公共衛生意涵</h2></header><div class="logic finding"><b>討論與限制</b><p>{REPORT_DISCUSSION}</p></div><div class="logic next"><b>公共衛生行動</b><p>{REPORT_ACTIONS}</p></div></section><footer>疾病、人物、場所、檢驗、事件與數值均為虛構合成教學設定。</footer></body></html>'''

interactive_path = OUTPUT_ROOT / "jinyuliang_investigation_report_interactive.html"
if BUILD_INTERACTIVE_HTML:
    interactive_path.write_text(interactive_html, encoding="utf-8")
    print("互動 HTML：", interactive_path.resolve())


## 13｜下載本次講師輸出

這一格只打包分析輸出，不包含網站原始碼或資料生成器。ZIP 內含 SVG、PNG、圖表索引、Word、列印版 PDF 與互動 HTML。

### 交付前檢查

建議先下載並解壓 ZIP，確認報告文件能開啟、八張圖都同時有 PNG 與 SVG、互動 HTML 可以離線載入，並保留本次執行後的 Notebook 作為可追溯紀錄。


In [ ]:
# ─────────────────────────────────────────────────────────────
# 13.1｜把講師輸出資料夾壓成一個 ZIP 並在 Colab 下載
# ─────────────────────────────────────────────────────────────
import shutil

archive_path = Path(shutil.make_archive(
    "jinyuliang_instructor_outputs",
    "zip",
    root_dir=OUTPUT_ROOT,
))
print("講師輸出包：", archive_path.resolve())

try:
    from google.colab import files
    files.download(str(archive_path))
except ImportError:
    print("目前不是 Colab；請直接開啟上方 ZIP 路徑。")


## 講師執行驗收

- 固定資料仍為 312 人；主要病例 77 人。
- `0.2` 沒有資料生成、seed 或 DGP 選項。
- 每一幕先建立結果表，再由 `publication` 函式畫圖。
- 八張出版圖都同時存在 `.svg` 與 `.png`，而且主要數值直接標在圖面。
- `chart_index.csv` 有八列且寫明各圖分母。
- Word 與列印版 PDF 使用出版 PNG，不依賴 hover。
- 互動 HTML 最後才建立，並沿用同一批結果表。
- canonical 預設下：翡翠凝露羹粗 OR 約 10.53、調整 OR 約 8.91；合歡桂露飲調整 CI 包含 1。

### 建議的參數實驗

1. 把 `PUBLICATION_EPI_BIN_HOURS` 改為 2 或 6，只重跑 `4.1`、`4.2` 與報告輸出。
2. 把 `CASE_DEFINITION` 改為 `sensitive`，從 `2.1` 往下重跑，觀察所有分母與效果如何連動。
3. 調換 `CANDIDATE_FOODS` 第二項，從 `7.1` 往下重跑，確認模型規格與公衛假說如何對應。
